<a href="https://colab.research.google.com/github/AishvaryaGovindaraju/DL-XAI/blob/main/DL_XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [19]:
from pathlib import Path

folders = [
    "project/data/raw",
    "project/data/processed",
    "project/src/preprocessing",
    "project/src/models",
    "project/src/xai",
    "project/src/evaluation",
    "project/src/statistics",
    "project/notebooks",
    "project/results/xai_outputs",
    "project/figures",
    "project/paper"
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project folder structure created successfully.")

Project folder structure created successfully.


In [20]:
!pip install -q torch shap lime dice-ml xgboost scikit-learn pandas numpy scipy scikit-posthocs pingouin matplotlib seaborn kaggle

In [21]:
import torch
import shap
import lime
import dice_ml
import xgboost
import sklearn
import pandas
import numpy
import scipy
import scikit_posthocs
import pingouin
import matplotlib
import seaborn

print("All required libraries imported successfully.")
print("PyTorch version:", torch.__version__)

All required libraries imported successfully.
PyTorch version: 2.11.0+cpu


In [22]:
import shutil
from pathlib import Path

# Create the required folder
Path("project/data/raw").mkdir(parents=True, exist_ok=True)

# Change this filename if Colab shows a slightly different uploaded name
source = "/content/diabetes_binary_health_indicators_BRFSS2015.csv"
destination = "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"

shutil.copy(source, destination)

print("Dataset copied successfully!")
print("Location:", destination)

Dataset copied successfully!
Location: project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv


In [23]:
import pandas as pd

file_path = "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (253680, 22)

Columns:
['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


In [24]:
# Check the target distribution

target_counts = df["Diabetes_binary"].value_counts().sort_index()
target_percent = df["Diabetes_binary"].value_counts(normalize=True).sort_index() * 100

print("Diabetes_binary distribution:")
print(target_counts)

print("\nPercentage distribution:")
print(target_percent)

print("\nClass labels:")
print("0 = No diabetes")
print("1 = Diabetes")

Diabetes_binary distribution:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Percentage distribution:
Diabetes_binary
0.0    86.066698
1.0    13.933302
Name: proportion, dtype: float64

Class labels:
0 = No diabetes
1 = Diabetes


In [25]:
# Check for missing values in every column

missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values in dataset:", missing_values.sum())

if missing_values.sum() == 0:
    print("✓ No missing values found.")
else:
    print("⚠ Missing values found.")

Missing values per column:
Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Total missing values in dataset: 0
✓ No missing values found.


In [26]:
# Check data types and number of unique values

print("DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n\nUNIQUE VALUES PER COLUMN")
print("=" * 50)
print(df.nunique().sort_values())

DATA TYPES
Diabetes_binary         float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object


UNIQUE VALUES PER COLUMN
Diabetes_binary          2
HighBP                   2
HighChol                 2
CholCheck                2
Smoker                   2
Stroke                   2
HeartDiseaseorAttack     2
PhysActivity             2
AnyHealthcare            2
F

In [27]:
# Inspect the actual unique values of important features

features_to_check = [
    "HighBP",
    "Sex",
    "GenHlth",
    "Age",
    "Education",
    "Income",
    "BMI",
    "MentHlth",
    "PhysHlth"
]

for feature in features_to_check:
    print(f"\n{feature}")
    print("-" * 40)
    print(sorted(df[feature].unique()))


HighBP
----------------------------------------
[np.float64(0.0), np.float64(1.0)]

Sex
----------------------------------------
[np.float64(0.0), np.float64(1.0)]

GenHlth
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

Age
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0)]

Education
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]

Income
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0)]

BMI
----------------------------------------
[np.float64(12.0), np.float64(13.0), np.

In [28]:
# Descriptive statistics for all columns

pd.set_option("display.max_columns", None)

print("DESCRIPTIVE STATISTICS")
print("=" * 80)

print(df.describe().T)

DESCRIPTIVE STATISTICS
                         count       mean       std   min   25%   50%   75%  \
Diabetes_binary       253680.0   0.139333  0.346294   0.0   0.0   0.0   0.0   
HighBP                253680.0   0.429001  0.494934   0.0   0.0   0.0   1.0   
HighChol              253680.0   0.424121  0.494210   0.0   0.0   0.0   1.0   
CholCheck             253680.0   0.962670  0.189571   0.0   1.0   1.0   1.0   
BMI                   253680.0  28.382364  6.608694  12.0  24.0  27.0  31.0   
Smoker                253680.0   0.443169  0.496761   0.0   0.0   0.0   1.0   
Stroke                253680.0   0.040571  0.197294   0.0   0.0   0.0   0.0   
HeartDiseaseorAttack  253680.0   0.094186  0.292087   0.0   0.0   0.0   0.0   
PhysActivity          253680.0   0.756544  0.429169   0.0   1.0   1.0   1.0   
Fruits                253680.0   0.634256  0.481639   0.0   0.0   1.0   1.0   
Veggies               253680.0   0.811420  0.391175   0.0   1.0   1.0   1.0   
HvyAlcoholConsump     253680.

In [29]:
# Correlation of each feature with the diabetes target

correlations = df.corr(numeric_only=True)["Diabetes_binary"].drop("Diabetes_binary")

correlations = correlations.sort_values(ascending=False)

print("Feature correlation with Diabetes_binary:")
print("=" * 60)
print(correlations)

Feature correlation with Diabetes_binary:
GenHlth                 0.293569
HighBP                  0.263129
DiffWalk                0.218344
BMI                     0.216843
HighChol                0.200276
Age                     0.177442
HeartDiseaseorAttack    0.177282
PhysHlth                0.171337
Stroke                  0.105816
MentHlth                0.069315
CholCheck               0.064761
Smoker                  0.060789
NoDocbcCost             0.031433
Sex                     0.031430
AnyHealthcare           0.016255
Fruits                 -0.040779
Veggies                -0.056584
HvyAlcoholConsump      -0.057056
PhysActivity           -0.118133
Education              -0.124456
Income                 -0.163919
Name: Diabetes_binary, dtype: float64


In [30]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"]

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split: divide the 30% temporary data equally
# This gives 15% validation and 15% test overall
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Dataset split completed.\n")

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Dataset split completed.

Training set: (177576, 21) (177576,)
Validation set: (38052, 21) (38052,)
Test set: (38052, 21) (38052,)


In [31]:
stratify=y

In [32]:
# Verify class distribution in each split

def show_distribution(name, target):
    counts = target.value_counts().sort_index()
    percentages = target.value_counts(normalize=True).sort_index() * 100

    print(f"\n{name}")
    print("-" * 50)

    for class_value in counts.index:
        print(
            f"Class {int(class_value)}: "
            f"{counts[class_value]:,} samples "
            f"({percentages[class_value]:.2f}%)"
        )

show_distribution("TRAINING SET", y_train)
show_distribution("VALIDATION SET", y_val)
show_distribution("TEST SET", y_test)


TRAINING SET
--------------------------------------------------
Class 0: 152,834 samples (86.07%)
Class 1: 24,742 samples (13.93%)

VALIDATION SET
--------------------------------------------------
Class 0: 32,750 samples (86.07%)
Class 1: 5,302 samples (13.93%)

TEST SET
--------------------------------------------------
Class 0: 32,750 samples (86.07%)
Class 1: 5,302 samples (13.93%)


In [33]:
from sklearn.preprocessing import StandardScaler

# Features specified by the project for standardization
scaled_features = [
    "BMI",
    "MentHlth",
    "PhysHlth",
    "Age"
]

# Create the scaler
scaler = StandardScaler()

# Fit ONLY on the training data
scaler.fit(X_train[scaled_features])

print("Scaler fitted successfully.")
print("\nFeatures being standardized:")
print(scaled_features)

print("\nTraining means learned by scaler:")
print(scaler.mean_)

print("\nTraining standard deviations learned by scaler:")
print(scaler.scale_)

Scaler fitted successfully.

Features being standardized:
['BMI', 'MentHlth', 'PhysHlth', 'Age']

Training means learned by scaler:
[28.37629522  3.19317926  4.24358021  8.03167095]

Training standard deviations learned by scaler:
[6.60785624 7.42707453 8.71942756 3.05035369]


In [34]:
scaler.fit(X_train[scaled_features])

StandardScaler()

In [35]:
# Create copies so the original split data remains unchanged
X_train_processed = X_train.copy()
X_val_processed = X_val.copy()
X_test_processed = X_test.copy()

# Apply the training-fitted scaler to all three datasets
X_train_processed[scaled_features] = scaler.transform(
    X_train[scaled_features]
)

X_val_processed[scaled_features] = scaler.transform(
    X_val[scaled_features]
)

X_test_processed[scaled_features] = scaler.transform(
    X_test[scaled_features]
)

print("Preprocessing transformation completed successfully.")
print("\nTraining shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Test shape:", X_test_processed.shape)

Preprocessing transformation completed successfully.

Training shape: (177576, 21)
Validation shape: (38052, 21)
Test shape: (38052, 21)


In [36]:
# Verify that the training data was standardized correctly

print("TRAINING SET AFTER STANDARDIZATION")
print("=" * 60)

for feature in scaled_features:
    print(
        f"{feature:10s} | "
        f"mean = {X_train_processed[feature].mean():.6f} | "
        f"std = {X_train_processed[feature].std():.6f}"
    )

TRAINING SET AFTER STANDARDIZATION
BMI        | mean = 0.000000 | std = 1.000003
MentHlth   | mean = -0.000000 | std = 1.000003
PhysHlth   | mean = -0.000000 | std = 1.000003
Age        | mean = 0.000000 | std = 1.000003


In [37]:
import joblib
from pathlib import Path

# Create model/preprocessing directory
Path("project/src/models").mkdir(parents=True, exist_ok=True)

# Save the fitted scaler
scaler_path = "project/src/models/scaler.pkl"

joblib.dump(scaler, scaler_path)

print("Scaler saved successfully!")
print("Saved to:", scaler_path)

Scaler saved successfully!
Saved to: project/src/models/scaler.pkl


In [38]:
# Verify that the saved scaler can be loaded

loaded_scaler = joblib.load(scaler_path)

print("Saved scaler loaded successfully!")
print("Scaled features:", loaded_scaler.feature_names_in_)

Saved scaler loaded successfully!
Scaled features: ['BMI' 'MentHlth' 'PhysHlth' 'Age']


In [39]:
import numpy as np

# Convert target labels to float32
y_train_processed = y_train.to_numpy(dtype=np.float32)
y_val_processed = y_val.to_numpy(dtype=np.float32)
y_test_processed = y_test.to_numpy(dtype=np.float32)

print("Labels prepared successfully.")

print("\nTraining labels:")
print("Shape:", y_train_processed.shape)
print("Data type:", y_train_processed.dtype)
print("Unique values:", np.unique(y_train_processed))

print("\nValidation labels:")
print("Shape:", y_val_processed.shape)
print("Data type:", y_val_processed.dtype)
print("Unique values:", np.unique(y_val_processed))

print("\nTest labels:")
print("Shape:", y_test_processed.shape)
print("Data type:", y_test_processed.dtype)
print("Unique values:", np.unique(y_test_processed))

Labels prepared successfully.

Training labels:
Shape: (177576,)
Data type: float32
Unique values: [0. 1.]

Validation labels:
Shape: (38052,)
Data type: float32
Unique values: [0. 1.]

Test labels:
Shape: (38052,)
Data type: float32
Unique values: [0. 1.]


In [40]:
# Convert processed feature DataFrames to NumPy arrays

X_train_np = X_train_processed.to_numpy(dtype=np.float32)
X_val_np = X_val_processed.to_numpy(dtype=np.float32)
X_test_np = X_test_processed.to_numpy(dtype=np.float32)

print("Feature arrays created successfully.")

print("\nTraining:")
print("Shape:", X_train_np.shape)
print("Data type:", X_train_np.dtype)

print("\nValidation:")
print("Shape:", X_val_np.shape)
print("Data type:", X_val_np.dtype)

print("\nTest:")
print("Shape:", X_test_np.shape)
print("Data type:", X_test_np.dtype)

Feature arrays created successfully.

Training:
Shape: (177576, 21)
Data type: float32

Validation:
Shape: (38052, 21)
Data type: float32

Test:
Shape: (38052, 21)
Data type: float32


In [41]:
# Final preprocessing sanity check

print("FINAL PREPROCESSING SANITY CHECK")
print("=" * 60)

# Check NaN values
print("\nNaN values:")
print("Train:", np.isnan(X_train_np).sum())
print("Validation:", np.isnan(X_val_np).sum())
print("Test:", np.isnan(X_test_np).sum())

# Check infinite values
print("\nInfinite values:")
print("Train:", np.isinf(X_train_np).sum())
print("Validation:", np.isinf(X_val_np).sum())
print("Test:", np.isinf(X_test_np).sum())

# Check overall numerical ranges
print("\nOverall feature ranges:")
print("Train min:", X_train_np.min())
print("Train max:", X_train_np.max())
print("Validation min:", X_val_np.min())
print("Validation max:", X_val_np.max())
print("Test min:", X_test_np.min())
print("Test max:", X_test_np.max())

FINAL PREPROCESSING SANITY CHECK

NaN values:
Train: 0
Validation: 0
Test: 0

Infinite values:
Train: 0
Validation: 0
Test: 0

Overall feature ranges:
Train min: -2.4783068
Train max: 10.536504
Validation min: -2.4783068
Validation max: 10.0824995
Test min: -2.4783068
Test max: 10.536504


In [42]:
import torch

# Convert feature arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)

# Convert labels to PyTorch tensors
y_train_tensor = torch.tensor(y_train_processed, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_processed, dtype=torch.float32)

print("PyTorch tensors created successfully.")

print("\nTraining:")
print("X:", X_train_tensor.shape, X_train_tensor.dtype)
print("y:", y_train_tensor.shape, y_train_tensor.dtype)

print("\nValidation:")
print("X:", X_val_tensor.shape, X_val_tensor.dtype)
print("y:", y_val_tensor.shape, y_val_tensor.dtype)

print("\nTest:")
print("X:", X_test_tensor.shape, X_test_tensor.dtype)
print("y:", y_test_tensor.shape, y_test_tensor.dtype)

PyTorch tensors created successfully.

Training:
X: torch.Size([177576, 21]) torch.float32
y: torch.Size([177576]) torch.float32

Validation:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32

Test:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32


In [43]:
import torch.nn as nn

class DiabetesDNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(21, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Second hidden layer
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Third hidden layer
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output layer
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [44]:
# Create the DNN model
model = DiabetesDNN()

print(model)

DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [45]:
# Count trainable parameters

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", total_params)

print("\nParameters by layer:")
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(f"{name:30s} {parameter.numel():,}")

Trainable parameters: 13185

Parameters by layer:
network.0.weight               2,688
network.0.bias                 128
network.3.weight               8,192
network.3.bias                 64
network.6.weight               2,048
network.6.bias                 32
network.9.weight               32
network.9.bias                 1


In [46]:
import numpy as np

# Count classes in the training set
class_counts = np.bincount(y_train_processed.astype(int))

# Calculate balanced class weights
num_samples = len(y_train_processed)
num_classes = len(class_counts)

class_weights = num_samples / (num_classes * class_counts)

print("Training class counts:")
print("Class 0:", class_counts[0])
print("Class 1:", class_counts[1])

print("\nCalculated class weights:")
print("Class 0:", class_weights[0])
print("Class 1:", class_weights[1])

Training class counts:
Class 0: 152834
Class 1: 24742

Calculated class weights:
Class 0: 0.5809440307784917
Class 1: 3.5885538760003235


In [47]:
import torch
import torch.nn as nn

# Convert the positive-class weight to a PyTorch tensor
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

# Weighted binary cross-entropy loss
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("Weighted loss function created successfully.")
print("Positive-class weight:", pos_weight.item())
print("Loss function:", criterion)

Weighted loss function created successfully.
Positive-class weight: 3.5885539054870605
Loss function: BCEWithLogitsLoss()


In [48]:
import torch.optim as optim

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Optimizer created successfully.")
print("Optimizer:", optimizer)
print("Learning rate:", 1e-3)

Optimizer created successfully.
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [49]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# Create DataLoaders
batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DataLoaders created successfully.")

print("\nBatch size:", batch_size)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders created successfully.

Batch size: 256
Training batches: 694
Validation batches: 149
Test batches: 149


In [50]:
# Inspect one batch from the training DataLoader

X_batch, y_batch = next(iter(train_loader))

print("ONE TRAINING BATCH")
print("=" * 60)

print("Feature batch shape:", X_batch.shape)
print("Feature data type:", X_batch.dtype)

print("\nLabel batch shape:", y_batch.shape)
print("Label data type:", y_batch.dtype)

print("\nFirst patient's features:")
print(X_batch[0])

print("\nFirst 10 labels:")
print(y_batch[:10])

ONE TRAINING BATCH
Feature batch shape: torch.Size([256, 21])
Feature data type: torch.float32

Label batch shape: torch.Size([256])
Label data type: torch.float32

First patient's features:
tensor([ 0.0000,  0.0000,  1.0000, -0.6623,  1.0000,  0.0000,  0.0000,  1.0000,
         0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  3.0000,  0.9165,  0.0868,
         0.0000,  0.0000,  0.3174,  6.0000,  8.0000])

First 10 labels:
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])


In [51]:
# Put the model in training mode
model.train()

# Perform one forward pass
logits = model(X_batch)

print("FORWARD PASS")
print("=" * 60)

print("Input shape:", X_batch.shape)
print("Output shape:", logits.shape)

print("\nFirst 10 raw logits:")
print(logits[:10].squeeze())

# Convert logits to probabilities for inspection
probabilities = torch.sigmoid(logits)

print("\nFirst 10 probabilities:")
print(probabilities[:10].squeeze())

FORWARD PASS
Input shape: torch.Size([256, 21])
Output shape: torch.Size([256, 1])

First 10 raw logits:
tensor([ 0.2431,  0.0438, -0.0259,  0.1123,  0.0144,  0.1372,  0.1251,  0.0987,
         0.0459,  0.0492], grad_fn=<SqueezeBackward0>)

First 10 probabilities:
tensor([0.5605, 0.5110, 0.4935, 0.5280, 0.5036, 0.5343, 0.5312, 0.5247, 0.5115,
        0.5123], grad_fn=<SqueezeBackward0>)


In [52]:
# Calculate the initial loss for the current batch

# Remove the final dimension from logits
# [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate weighted binary cross-entropy loss
initial_loss = criterion(batch_logits, y_batch)

print("INITIAL LOSS")
print("=" * 60)
print("Loss:", initial_loss.item())

INITIAL LOSS
Loss: 0.9511492252349854


In [53]:
# Perform one complete training step

# Make sure the model is in training mode
model.train()

# Clear gradients from any previous step
optimizer.zero_grad()

# Forward pass
logits = model(X_batch)

# Remove the final dimension: [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate loss
loss = criterion(batch_logits, y_batch)

# Backpropagation
loss.backward()

# Update model parameters
optimizer.step()

print("ONE TRAINING STEP COMPLETED")
print("=" * 60)
print("Loss before parameter update:", loss.item())

ONE TRAINING STEP COMPLETED
Loss before parameter update: 0.9406346082687378


In [54]:
# Check whether the model parameters were updated

print("PARAMETER UPDATE CHECK")
print("=" * 60)

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(
            f"{name:30s} | "
            f"mean = {parameter.data.mean().item():.6f} | "
            f"grad mean = {parameter.grad.mean().item():.6f}"
        )

PARAMETER UPDATE CHECK
network.0.weight               | mean = 0.005111 | grad mean = -0.000041
network.0.bias                 | mean = -0.009275 | grad mean = -0.000072
network.3.weight               | mean = 0.000388 | grad mean = -0.000110
network.3.bias                 | mean = 0.012954 | grad mean = -0.000226
network.6.weight               | mean = -0.001472 | grad mean = 0.000298
network.6.bias                 | mean = -0.002238 | grad mean = 0.001196
network.9.weight               | mean = 0.013723 | grad mean = 0.025389
network.9.bias                 | mean = 0.127543 | grad mean = 0.250628


In [55]:
# Reset the model before real training

model = DiabetesDNN()

# Recreate the weighted loss
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Recreate the optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Model reset successfully.")
print("\nArchitecture:")
print(model)

print("\nTrainable parameters:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

print("\nOptimizer:")
print(optimizer.__class__.__name__)

print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Positive-class weight:", pos_weight.item())

Model reset successfully.

Architecture:
DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)

Trainable parameters:
13185

Optimizer:
Adam
Learning rate: 0.001
Positive-class weight: 3.5885539054870605


In [56]:
from sklearn.metrics import roc_auc_score
import copy
import numpy as np
import torch

# Training configuration
max_epochs = 100
patience = 10

# Store training history
history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

# Early stopping variables
best_val_auc = -np.inf
epochs_without_improvement = 0
best_model_state = None

print("Starting DNN training...")
print("=" * 70)

for epoch in range(max_epochs):

    # ============================================================
    # TRAINING
    # ============================================================

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(X_batch).squeeze(1)

        # Calculate weighted loss
        loss = criterion(logits, y_batch)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Accumulate loss
        batch_size_actual = X_batch.size(0)
        running_train_loss += loss.item() * batch_size_actual
        train_samples += batch_size_actual

    epoch_train_loss = running_train_loss / train_samples

    # ============================================================
    # VALIDATION
    # ============================================================

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    val_probabilities = []
    val_true_labels = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            # Forward pass
            logits = model(X_batch).squeeze(1)

            # Validation loss
            loss = criterion(logits, y_batch)

            # Convert logits to probabilities
            probabilities = torch.sigmoid(logits)

            # Accumulate validation loss
            batch_size_actual = X_batch.size(0)
            running_val_loss += loss.item() * batch_size_actual
            val_samples += batch_size_actual

            # Store predictions and true labels
            val_probabilities.extend(
                probabilities.cpu().numpy()
            )

            val_true_labels.extend(
                y_batch.cpu().numpy()
            )

    epoch_val_loss = running_val_loss / val_samples

    # Calculate validation ROC-AUC
    epoch_val_auc = roc_auc_score(
        val_true_labels,
        val_probabilities
    )

    # Store history
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_auc"].append(epoch_val_auc)

    # ============================================================
    # EARLY STOPPING / CHECKPOINTING
    # ============================================================

    if epoch_val_auc > best_val_auc:

        # Validation AUC improved
        best_val_auc = epoch_val_auc
        epochs_without_improvement = 0

        # Save a copy of the best model parameters
        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        checkpoint_status = " ← BEST"

    else:

        # No improvement
        epochs_without_improvement += 1

        checkpoint_status = (
            f" | patience: "
            f"{epochs_without_improvement}/{patience}"
        )

    # ============================================================
    # PRINT EPOCH RESULTS
    # ============================================================

    print(
        f"Epoch {epoch + 1:03d}/{max_epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val ROC-AUC: {epoch_val_auc:.4f}"
        f"{checkpoint_status}"
    )

    # ============================================================
    # EARLY STOPPING
    # ============================================================

    if epochs_without_improvement >= patience:

        print("\nEarly stopping triggered.")
        print(
            f"No validation ROC-AUC improvement for "
            f"{patience} consecutive epochs."
        )

        break


# ================================================================
# RESTORE BEST MODEL
# ================================================================

if best_model_state is not None:

    model.load_state_dict(best_model_state)

    print("\nBest model restored successfully.")
    print(f"Best validation ROC-AUC: {best_val_auc:.4f}")

print("\nTraining completed.")

Starting DNN training...
Epoch 001/100 | Train Loss: 0.6994 | Val Loss: 0.6707 | Val ROC-AUC: 0.8228 ← BEST
Epoch 002/100 | Train Loss: 0.6759 | Val Loss: 0.6690 | Val ROC-AUC: 0.8246 ← BEST
Epoch 003/100 | Train Loss: 0.6728 | Val Loss: 0.6675 | Val ROC-AUC: 0.8253 ← BEST
Epoch 004/100 | Train Loss: 0.6705 | Val Loss: 0.6658 | Val ROC-AUC: 0.8264 ← BEST
Epoch 005/100 | Train Loss: 0.6683 | Val Loss: 0.6656 | Val ROC-AUC: 0.8265 ← BEST
Epoch 006/100 | Train Loss: 0.6684 | Val Loss: 0.6629 | Val ROC-AUC: 0.8267 ← BEST
Epoch 007/100 | Train Loss: 0.6677 | Val Loss: 0.6630 | Val ROC-AUC: 0.8272 ← BEST
Epoch 008/100 | Train Loss: 0.6671 | Val Loss: 0.6645 | Val ROC-AUC: 0.8264 | patience: 1/10
Epoch 009/100 | Train Loss: 0.6656 | Val Loss: 0.6635 | Val ROC-AUC: 0.8271 | patience: 2/10
Epoch 010/100 | Train Loss: 0.6661 | Val Loss: 0.6631 | Val ROC-AUC: 0.8272 | patience: 3/10
Epoch 011/100 | Train Loss: 0.6652 | Val Loss: 0.6642 | Val ROC-AUC: 0.8276 ← BEST
Epoch 012/100 | Train Loss: 0.66

In [60]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Put the restored best model into evaluation mode
model.eval()

test_probabilities = []
test_true_labels = []

# Generate predictions without calculating gradients
with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # Forward pass
        logits = model(X_batch).squeeze(1)

        # Convert logits to probabilities
        probabilities = torch.sigmoid(logits)

        # Store predictions and true labels
        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

        test_true_labels.extend(
            y_batch.cpu().numpy()
        )

# Convert to NumPy arrays
test_probabilities = np.array(test_probabilities)
test_true_labels = np.array(test_true_labels)

print("TEST PREDICTIONS GENERATED")
print("=" * 60)

print("Number of predictions:", len(test_probabilities))
print("Number of true labels:", len(test_true_labels))

print("\nProbability range:")
print("Minimum:", test_probabilities.min())
print("Maximum:", test_probabilities.max())

print("\nFirst 10 test probabilities:")
print(test_probabilities[:10])

print("\nFirst 10 true labels:")
print(test_true_labels[:10])

TEST PREDICTIONS GENERATED
Number of predictions: 38052
Number of true labels: 38052

Probability range:
Minimum: 0.00010720247
Maximum: 0.92266446

First 10 test probabilities:
[0.51072556 0.16627327 0.5427737  0.46732858 0.32918245 0.14756724
 0.26985973 0.07064747 0.04576432 0.41812947]

First 10 true labels:
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [61]:
# ============================================================
# FINAL TEST METRICS
# ============================================================

# Classification threshold
threshold = 0.5

# Convert probabilities into binary predictions
test_predictions = (
    test_probabilities >= threshold
).astype(int)

# ------------------------------------------------------------
# Threshold-independent metrics
# ------------------------------------------------------------

test_roc_auc = roc_auc_score(
    test_true_labels,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_true_labels,
    test_probabilities
)

# ------------------------------------------------------------
# Threshold-dependent metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
)

test_precision = precision_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

test_cm = confusion_matrix(
    test_true_labels,
    test_predictions
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("FINAL DNN TEST RESULTS")
print("=" * 60)

print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\nConfusion Matrix Interpretation:")
print(f"TN = {test_cm[0, 0]}")
print(f"FP = {test_cm[0, 1]}")
print(f"FN = {test_cm[1, 0]}")
print(f"TP = {test_cm[1, 1]}")

FINAL DNN TEST RESULTS
ROC-AUC   : 0.8298
PR-AUC    : 0.4264
Accuracy  : 0.7954
Precision : 0.3667
Recall    : 0.6445
F1-score  : 0.4675

Confusion Matrix:
[[26850  5900]
 [ 1885  3417]]

Confusion Matrix Interpretation:
TN = 26850
FP = 5900
FN = 1885
TP = 3417


In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class UA_DNN(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.drop2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(64, 32)
        self.drop3 = nn.Dropout(0.2)

        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):

        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.drop2(x)

        x = self.fc3(x)
        x = F.relu(x)
        x = self.drop3(x)

        x = self.fc4(x)

        return torch.sigmoid(x)


# Create the PRD-compliant model
model = UA_DNN(input_dim=21)

print("PRD-compliant DNN created successfully.")
print(model)

print("\nTrainable parameters:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

PRD-compliant DNN created successfully.
UA_DNN(
  (fc1): Linear(in_features=21, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (drop3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=32, out_features=1, bias=True)
)

Trainable parameters:
13569


In [63]:
# ============================================================
# PRD-COMPLIANT WEIGHTED BCELoss
# ============================================================

# Positive-class weight calculated from the training set
positive_weight = class_weights[1]

# Create per-sample weights
sample_weights = torch.where(
    y_train_tensor == 1.0,
    torch.tensor(positive_weight, dtype=torch.float32),
    torch.tensor(1.0, dtype=torch.float32)
)

# Weighted binary cross-entropy
criterion = nn.BCELoss(
    weight=sample_weights
)

print("PRD-compliant loss created successfully.")
print("Loss function:", criterion)
print("Positive-class weight:", positive_weight)

PRD-compliant loss created successfully.
Loss function: BCELoss()
Positive-class weight: 3.5885538760003235


In [64]:
# ============================================================
# PRD-COMPLIANT ADAM OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("PRD-compliant optimizer created successfully.")
print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])

PRD-compliant optimizer created successfully.
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [67]:
# ============================================================
# FIXED PRD-COMPLIANT BCELoss
# ============================================================

criterion = nn.BCELoss(reduction='none')

positive_weight = float(class_weights[1])

print("Base BCELoss created successfully.")
print("Positive-class weight:", positive_weight)
print("Loss reduction: none")

Base BCELoss created successfully.
Positive-class weight: 3.5885538760003235
Loss reduction: none


In [68]:
# ============================================================
# PRD-COMPLIANT DNN TRAINING LOOP — FIXED WEIGHTING
# ============================================================

from sklearn.metrics import roc_auc_score
import copy
import numpy as np
import torch


MAX_EPOCHS = 100
PATIENCE = 10

best_val_auc = -np.inf
best_model_state = None
patience_counter = 0

train_losses = []
val_losses = []
val_auc_history = []

print("Starting PRD-compliant DNN training...")
print("=" * 70)


for epoch in range(1, MAX_EPOCHS + 1):

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(X_batch).squeeze(1)

        # Individual BCE losses
        individual_losses = criterion(
            predictions,
            y_batch
        )

        # Assign class weight to positive samples
        batch_weights = torch.where(
            y_batch == 1.0,
            torch.tensor(
                positive_weight,
                dtype=torch.float32
            ),
            torch.tensor(
                1.0,
                dtype=torch.float32
            )
        )

        # Apply class weights
        weighted_loss = (
            individual_losses * batch_weights
        ).mean()

        # Backpropagation
        weighted_loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate batch loss
        batch_size = X_batch.size(0)

        running_train_loss += (
            weighted_loss.item() * batch_size
        )

        train_samples += batch_size

    epoch_train_loss = (
        running_train_loss / train_samples
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    val_probabilities = []
    val_true_labels = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            # Forward pass
            predictions = model(
                X_batch
            ).squeeze(1)

            # Individual validation losses
            individual_losses = criterion(
                predictions,
                y_batch
            )

            # Same class weighting
            batch_weights = torch.where(
                y_batch == 1.0,
                torch.tensor(
                    positive_weight,
                    dtype=torch.float32
                ),
                torch.tensor(
                    1.0,
                    dtype=torch.float32
                )
            )

            weighted_val_loss = (
                individual_losses * batch_weights
            ).mean()

            # Accumulate validation loss
            batch_size = X_batch.size(0)

            running_val_loss += (
                weighted_val_loss.item() * batch_size
            )

            val_samples += batch_size

            # Store probabilities
            val_probabilities.extend(
                predictions.cpu().numpy()
            )

            val_true_labels.extend(
                y_batch.cpu().numpy()
            )


    epoch_val_loss = (
        running_val_loss / val_samples
    )


    # ========================================================
    # VALIDATION ROC-AUC
    # ========================================================

    epoch_val_auc = roc_auc_score(
        val_true_labels,
        val_probabilities
    )


    # ========================================================
    # STORE HISTORY
    # ========================================================

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    val_auc_history.append(epoch_val_auc)


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    if epoch_val_auc > best_val_auc:

        best_val_auc = epoch_val_auc

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Val ROC-AUC: {epoch_val_auc:.4f} "
            f"← BEST"
        )

    else:

        patience_counter += 1

        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Val ROC-AUC: {epoch_val_auc:.4f} "
            f"| patience: "
            f"{patience_counter}/{PATIENCE}"
        )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")

        print(
            f"No validation ROC-AUC improvement "
            f"for {PATIENCE} consecutive epochs."
        )

        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

    print("\nBest model restored successfully.")

    print(
        f"Best validation ROC-AUC: "
        f"{best_val_auc:.4f}"
    )

print("\nTraining completed.")

Starting PRD-compliant DNN training...
Epoch 001/100 | Train Loss: 0.6892 | Val Loss: 0.6653 | Val ROC-AUC: 0.8254 ← BEST
Epoch 002/100 | Train Loss: 0.6726 | Val Loss: 0.6629 | Val ROC-AUC: 0.8272 ← BEST
Epoch 003/100 | Train Loss: 0.6678 | Val Loss: 0.6616 | Val ROC-AUC: 0.8276 ← BEST
Epoch 004/100 | Train Loss: 0.6662 | Val Loss: 0.6620 | Val ROC-AUC: 0.8274 | patience: 1/10
Epoch 005/100 | Train Loss: 0.6649 | Val Loss: 0.6634 | Val ROC-AUC: 0.8278 ← BEST
Epoch 006/100 | Train Loss: 0.6650 | Val Loss: 0.6614 | Val ROC-AUC: 0.8282 ← BEST
Epoch 007/100 | Train Loss: 0.6633 | Val Loss: 0.6619 | Val ROC-AUC: 0.8283 ← BEST
Epoch 008/100 | Train Loss: 0.6634 | Val Loss: 0.6607 | Val ROC-AUC: 0.8284 ← BEST
Epoch 009/100 | Train Loss: 0.6626 | Val Loss: 0.6608 | Val ROC-AUC: 0.8285 ← BEST
Epoch 010/100 | Train Loss: 0.6616 | Val Loss: 0.6598 | Val ROC-AUC: 0.8288 ← BEST
Epoch 011/100 | Train Loss: 0.6619 | Val Loss: 0.6607 | Val ROC-AUC: 0.8290 ← BEST
Epoch 012/100 | Train Loss: 0.6620 | V

In [69]:
# ============================================================
# FINAL TEST EVALUATION — PRD-COMPLIANT DNN
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Make absolutely sure the best checkpoint is loaded
model.load_state_dict(best_model_state)

# Evaluation mode
model.eval()

test_probabilities = []
test_true_labels = []

# Generate final test probabilities
with torch.no_grad():

    for X_batch, y_batch in test_loader:

        predictions = model(
            X_batch
        ).squeeze(1)

        test_probabilities.extend(
            predictions.cpu().numpy()
        )

        test_true_labels.extend(
            y_batch.cpu().numpy()
        )

# Convert to NumPy
test_probabilities = np.array(
    test_probabilities
)

test_true_labels = np.array(
    test_true_labels
)

# ============================================================
# CLASSIFICATION THRESHOLD
# ============================================================

threshold = 0.5

test_predictions = (
    test_probabilities >= threshold
).astype(int)

# ============================================================
# METRICS
# ============================================================

test_roc_auc = roc_auc_score(
    test_true_labels,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_true_labels,
    test_probabilities
)

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
)

test_precision = precision_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_cm = confusion_matrix(
    test_true_labels,
    test_predictions
)

# ============================================================
# DISPLAY
# ============================================================

print("FINAL PRD-COMPLIANT DNN TEST RESULTS")
print("=" * 60)

print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\nConfusion Matrix Interpretation:")
print(f"TN = {test_cm[0, 0]}")
print(f"FP = {test_cm[0, 1]}")
print(f"FN = {test_cm[1, 0]}")
print(f"TP = {test_cm[1, 1]}")

print("\nProbability range:")
print(f"Minimum = {test_probabilities.min():.6f}")
print(f"Maximum = {test_probabilities.max():.6f}")

FINAL PRD-COMPLIANT DNN TEST RESULTS
ROC-AUC   : 0.8304
PR-AUC    : 0.4263
Accuracy  : 0.7971
Precision : 0.3690
Recall    : 0.6426
F1-score  : 0.4688

Confusion Matrix:
[[26925  5825]
 [ 1895  3407]]

Confusion Matrix Interpretation:
TN = 26925
FP = 5825
FN = 1895
TP = 3407

Probability range:
Minimum = 0.000109
Maximum = 0.951618


In [70]:
# ============================================================
# MC DROPOUT SETUP
# ============================================================

import torch.nn as nn

# Load the best DNN checkpoint again
model.load_state_dict(best_model_state)

# Start with the entire model in evaluation mode
model.eval()

# Activate ONLY dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

print("MC Dropout mode configured successfully.")
print("=" * 60)

# Verify the behavior of each layer
for name, module in model.named_modules():

    if isinstance(module, nn.Dropout):
        print(
            f"{name}: Dropout ACTIVE | "
            f"p = {module.p}"
        )

    elif isinstance(module, nn.BatchNorm1d):
        print(
            f"{name}: BatchNorm EVAL | "
            f"training = {module.training}"
        )

MC Dropout mode configured successfully.
bn1: BatchNorm EVAL | training = False
drop1: Dropout ACTIVE | p = 0.3
bn2: BatchNorm EVAL | training = False
drop2: Dropout ACTIVE | p = 0.3
drop3: Dropout ACTIVE | p = 0.2


In [71]:
# ============================================================
# MC DROPOUT INFERENCE — T = 50
# ============================================================

import numpy as np
import torch

# Number of stochastic forward passes
T = 50

# Make sure best checkpoint is loaded
model.load_state_dict(best_model_state)

# Evaluation mode first
model.eval()

# Activate ONLY Dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

# Store predictions from every stochastic pass
mc_predictions = []

print("Starting MC Dropout inference...")
print("=" * 60)
print(f"Number of stochastic passes (T): {T}")
print(f"Number of test samples: {len(test_true_labels)}")

# ------------------------------------------------------------
# Perform T stochastic forward passes
# ------------------------------------------------------------

with torch.no_grad():

    for t in range(T):

        pass_predictions = []

        for X_batch, _ in test_loader:

            predictions = model(
                X_batch
            ).squeeze(1)

            pass_predictions.extend(
                predictions.cpu().numpy()
            )

        pass_predictions = np.array(
            pass_predictions
        )

        mc_predictions.append(
            pass_predictions
        )

        print(
            f"Pass {t + 1:02d}/{T} completed | "
            f"Min: {pass_predictions.min():.6f} | "
            f"Max: {pass_predictions.max():.6f} | "
            f"Mean: {pass_predictions.mean():.6f}"
        )

# Convert to NumPy array
mc_predictions = np.array(
    mc_predictions
)

print("\nMC Dropout inference completed.")
print("=" * 60)

print(
    "MC prediction array shape:",
    mc_predictions.shape
)

print(
    "Expected shape:",
    (T, len(test_true_labels))
)

print(
    "Total stochastic predictions:",
    mc_predictions.size
)

Starting MC Dropout inference...
Number of stochastic passes (T): 50
Number of test samples: 38052
Pass 01/50 completed | Min: 0.000013 | Max: 0.981168 | Mean: 0.284780
Pass 02/50 completed | Min: 0.000005 | Max: 0.985518 | Mean: 0.284637
Pass 03/50 completed | Min: 0.000000 | Max: 0.986577 | Mean: 0.284792
Pass 04/50 completed | Min: 0.000009 | Max: 0.994473 | Mean: 0.284875
Pass 05/50 completed | Min: 0.000002 | Max: 0.988509 | Mean: 0.284689
Pass 06/50 completed | Min: 0.000010 | Max: 0.985008 | Mean: 0.284863
Pass 07/50 completed | Min: 0.000012 | Max: 0.985413 | Mean: 0.284786
Pass 08/50 completed | Min: 0.000003 | Max: 0.992556 | Mean: 0.284878
Pass 09/50 completed | Min: 0.000017 | Max: 0.978911 | Mean: 0.284943
Pass 10/50 completed | Min: 0.000006 | Max: 0.981705 | Mean: 0.284995
Pass 11/50 completed | Min: 0.000019 | Max: 0.985372 | Mean: 0.284789
Pass 12/50 completed | Min: 0.000009 | Max: 0.992593 | Mean: 0.284812
Pass 13/50 completed | Min: 0.000006 | Max: 0.989032 | Mean: 